# Context Manager

In [1]:
f = open('dummy-file.txt', 'r')
for line in f:
    print(line)

f.close()   # People will always forget this

hello


In [2]:
with open('dummy-file.txt', 'r') as f:
    for line in f:
        print(line)

# Now there is no need to manually close the file.

hello


The keyword `with` marks a context. We can define our own <strong>context managers</strong> as well! 

The idea is to create a context that requires some <strong>setup</strong> before starting and then some <strong>cleanup</strong> at the end. 

In [3]:
import time

def some_function():
    time.sleep(1)

In [4]:
%time some_function()

CPU times: user 873 μs, sys: 1.84 ms, total: 2.71 ms
Wall time: 1 s


But this is only available in jupyter notebook.
To do it in other python runtimes

In [18]:
start_time = int(round(time.time() * 1000))

some_function()

end_time = int(round(time.time() * 1000))
elapsed = end_time - start_time
print(f"Code took {elapsed} ms to run.")

Code took 1001 ms to run.


In [16]:
time.time() 

1782684741.4544063

Fairly simple but it's making our code look ugly and is a hassle. Let's do this using a context manager. It matches the pattern after all: 

* Do something at startup (Record start time)
* Perform some (unspecified) work
* Do something at end (Report time elapsed) 

In [24]:
from contextlib import contextmanager # import the decorator

@contextmanager   # mark the function as 'Context Manager' so we can use it by 'with' keyword
def timeit():
    start_time = int(round(time.time() * 1000))

    yield         # this is from the generator
                  # 'yield' will drop execution to acutal code with the 'with block'.       
    
    end_time = int(round(time.time() * 1000))
    elapsed = end_time - start_time
    print(f"Code took {elapsed} ms to run.")

In [25]:
with timeit():
    some_function()

Code took 1001 ms to run.


Now we can use same context manager with any function or code wusing `with timeit()`.

In [26]:
def another_funciton():
    time.sleep(0.5)

In [27]:
with timeit():
    another_funciton()

Code took 500 ms to run.


Clean code, it saves a lot of repitive code and time!

# Case Study: Temporary Directoreis
it's common to create temporary directories for files and delete them after you're done iwht tehm.

In [30]:
import tempfile
import shutil
import os

try:
    name = tempfile.mkdtemp()    # making temparory directory
    print(f"Created temp directory: {name}")

    filename = os.path.join(name, "somefile.txt")
    # Do some processing
    with open(filename, 'w') as f:
        print(f"Opened file: {filename}")
        f.write("Some Dummy Text")

finally:
    print(f"Deleteing directory: {name}")
    shutil.rmtree(name)

Created temp directory: /tmp/tmpji5wp5om
Opened file: /tmp/tmpji5wp5om/somefile.txt
Deleteing directory: /tmp/tmpji5wp5om


In [31]:
@contextmanager
def tempdir(filename):
    try:
        name = tempfile.mkdtemp()    # making temparory directory
        print(f"Created temp directory: {name}")
    
        filename = os.path.join(name, filename)
        # Do some processing
        with open(filename, 'w') as f:
            print(f"Opened file: {filename}")
            yield f
    
    finally:
        print(f"Deleteing directory: {name}")
        shutil.rmtree(name)

In [32]:
with tempdir('example.txt') as f:
    print(f"Doing something with file: {f.name}")

Created temp directory: /tmp/tmpp2uqg5yo
Opened file: /tmp/tmpp2uqg5yo/example.txt
Doing something with file: /tmp/tmpp2uqg5yo/example.txt
Deleteing directory: /tmp/tmpp2uqg5yo
